In [1]:
# ============================================================
# DATATHON 2026 — NON-LOGIN USER BEHAVIOR EDA
# DISK-SAFE FULL CODE
#
# Event-level summaries: FULL DATA
# Session-level dwell / interaction: SAMPLE SESSION, default 2%
#
# Outputs:
# - Tables: /kaggle/working/eda_non_login_behavior/tables
# - Charts: /kaggle/working/eda_non_login_behavior/figures
# ============================================================

import os
import glob
import gc
import sys
import shutil
import subprocess
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

try:
    import duckdb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

try:
    import plotly.express as px
    import plotly.graph_objects as go
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "plotly"])
    import plotly.express as px
    import plotly.graph_objects as go

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# ============================================================
# 0. CONFIG
# ============================================================

BASE_PATH_2 = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2"
EVENTS_PATH = os.path.join(BASE_PATH_2, "fact_user_events")

OUTPUT_DIR = "/kaggle/working/eda_non_login_behavior"
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
CACHE_DIR = os.path.join(OUTPUT_DIR, "cache")

CLEAN_OUTPUT = True

if CLEAN_OUTPUT and os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

PARTIAL_TIME_DIR = os.path.join(CACHE_DIR, "partial_time")
PARTIAL_EVENT_CAT_DIR = os.path.join(CACHE_DIR, "partial_event_category")
PARTIAL_CITY_CAT_DIR = os.path.join(CACHE_DIR, "partial_city_category")
PARTIAL_DEVICE_SURFACE_DIR = os.path.join(CACHE_DIR, "partial_device_surface")
PARTIAL_CONTACT_CITY_CAT_DIR = os.path.join(CACHE_DIR, "partial_contact_city_category")
PARTIAL_SESSION_DIR = os.path.join(CACHE_DIR, "partial_session_sample")

for d in [
    PARTIAL_TIME_DIR,
    PARTIAL_EVENT_CAT_DIR,
    PARTIAL_CITY_CAT_DIR,
    PARTIAL_DEVICE_SURFACE_DIR,
    PARTIAL_CONTACT_CITY_CAT_DIR,
    PARTIAL_SESSION_DIR,
]:
    os.makedirs(d, exist_ok=True)

EVENT_FILES = sorted(glob.glob(os.path.join(EVENTS_PATH, "*.parquet")))

BATCH_SIZE = 300_000
TOP_N = 30

# Session-level dùng sample để tránh đầy disk.
# 2 nghĩa là giữ 2% session.
# Nếu muốn kỹ hơn và disk còn nhiều: đổi thành 5.
# Nếu vẫn đầy disk: đổi thành 1.
SESSION_SAMPLE_MOD = 100
SESSION_SAMPLE_KEEP = 2

# Dwell cap để tránh outlier cực đoan.
DWELL_CAP_SEC = 1800

POSITIVE_EVENTS = {
    "view_phone",
    "contact_chat",
    "other_interaction",
    "contact_zalo",
    "contact_sms",
}

STRICT_CONTACT_EVENTS = {
    "view_phone",
    "contact_chat",
    "contact_zalo",
    "contact_sms",
}

EVENT_COLS = [
    "is_login",
    "user_id",
    "session_id",
    "event_id",
    "item_id",
    "city_name",
    "category",
    "event_type",
    "event_ts",
    "surface",
    "device",
    "dwell_time_sec",
    "is_contact",
    "date",
]

print("EVENTS_PATH:", EVENTS_PATH)
print("Event files:", len(EVENT_FILES))
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SESSION SAMPLE:", f"{SESSION_SAMPLE_KEEP}/{SESSION_SAMPLE_MOD} = {SESSION_SAMPLE_KEEP / SESSION_SAMPLE_MOD:.0%}")

if len(EVENT_FILES) == 0:
    raise FileNotFoundError(f"No parquet files found in {EVENTS_PATH}")

# ============================================================
# 1. HELPERS
# ============================================================

def clean_text_series(s, default="unknown"):
    return (
        s.astype("string")
         .fillna(default)
         .str.strip()
         .str.lower()
         .replace("", default)
    )

def normalize_login_group(s):
    x = clean_text_series(s)
    return np.where(x == "login", "login", "non_login")

def category_name_from_series(s):
    s_num = pd.to_numeric(s, errors="coerce").astype("Int64")
    mp = {
        1010: "1010_room_rental",
        1020: "1020_apartment",
        1030: "1030_house",
        1040: "1040_land_commercial",
        1050: "1050_new_project",
    }
    out = s_num.map(mp)
    return out.fillna("unknown_" + s_num.astype(str).fillna("NULL"))

def session_sample_mask(session_series, mod=100, keep=2):
    h = pd.util.hash_pandas_object(session_series.astype("string"), index=False).values
    return (h % mod) < keep

def write_parquet(df, path):
    df.to_parquet(path, index=False, compression="zstd")

def save_df(df, name, show_head=30):
    csv_path = os.path.join(TABLE_DIR, f"{name}.csv")
    parquet_path = os.path.join(TABLE_DIR, f"{name}.parquet")
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    try:
        df.to_parquet(parquet_path, index=False, compression="zstd")
    except Exception as e:
        print("[WARN] Cannot save parquet:", name, e)
    print("[SAVE TABLE]", csv_path)
    display(df.head(show_head))
    return df

def plotly_safe_df(df):
    return df.copy().astype(object).where(pd.notna(df), None)

chart_index = []

def save_fig(fig, name, title=None):
    html_path = os.path.join(FIG_DIR, f"{name}.html")
    fig.write_html(html_path)
    print("[SAVE FIG]", html_path)
    chart_index.append({
        "name": name,
        "title": title if title else name,
        "html_path": html_path,
    })
    fig.show()

def run_duck(query, filename=None, show_head=30, memory_limit="6GB"):
    con = duckdb.connect()
    con.execute("PRAGMA threads=2")
    con.execute(f"PRAGMA memory_limit='{memory_limit}'")
    con.execute("PRAGMA preserve_insertion_order=false")
    df = con.execute(query).df()
    con.close()
    if filename:
        return save_df(df, filename, show_head=show_head)
    return df

def make_label_event_category(df):
    df = df.copy()
    df["label"] = df["event_type_clean"].astype(str) + " | " + df["category_name"].astype(str)
    return df

def safe_rate(num, den):
    return num / den.replace(0, np.nan)

def list_glob(path):
    return sorted(glob.glob(path))

# ============================================================
# 2. MAP RAW EVENTS TO SMALL PARTIAL TABLES
# ============================================================

part_id = 0

for file_idx, fp in enumerate(EVENT_FILES):
    print(f"\n[MAP] file {file_idx + 1}/{len(EVENT_FILES)}: {os.path.basename(fp)}")

    pf = pq.ParquetFile(fp)

    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE, columns=EVENT_COLS)):
        df = batch.to_pandas()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        # -----------------------------
        # Clean base fields
        # -----------------------------
        df["is_login_group"] = normalize_login_group(df["is_login"])
        df["event_type_clean"] = clean_text_series(df["event_type"])
        df["city_clean"] = clean_text_series(df["city_name"])
        df["surface_clean"] = clean_text_series(df["surface"])
        df["device_clean"] = clean_text_series(df["device"])

        df["category"] = pd.to_numeric(df["category"], errors="coerce").astype("Int64")
        df["category_name"] = category_name_from_series(df["category"])

        df["active_date"] = pd.to_datetime(df["date"], errors="coerce").dt.floor("D")
        df["event_ts_clean"] = pd.to_datetime(df["event_ts"], errors="coerce")

        df["hour"] = df["event_ts_clean"].dt.hour.fillna(-1).astype("int16")
        df["dow_num"] = df["active_date"].dt.dayofweek.fillna(-1).astype("int16")
        df["dow_name"] = df["active_date"].dt.day_name().fillna("unknown")

        df["is_contact_int"] = pd.to_numeric(df["is_contact"], errors="coerce").fillna(0).astype("int8")
        df["is_positive_int"] = df["event_type_clean"].isin(POSITIVE_EVENTS).astype("int8")
        df["is_strict_contact_int"] = df["event_type_clean"].isin(STRICT_CONTACT_EVENTS).astype("int8")
        df["is_pageview_int"] = (df["event_type_clean"] == "pageview").astype("int8")

        dwell_raw = pd.to_numeric(df["dwell_time_sec"], errors="coerce").fillna(0).clip(lower=0)
        df["dwell_raw_sec"] = dwell_raw
        df["dwell_capped_sec"] = dwell_raw.clip(upper=DWELL_CAP_SEC)
        df["has_dwell"] = (df["dwell_raw_sec"] > 0).astype("int8")

        # -----------------------------
        # A. Time summary: date x hour x login group
        # -----------------------------
        time_base = df[df["active_date"].notna()].copy()

        if len(time_base) > 0:
            g = (
                time_base.groupby(
                    ["active_date", "hour", "dow_num", "dow_name", "is_login_group"],
                    observed=True,
                    dropna=False,
                )
                .agg(
                    event_rows=("event_type_clean", "size"),
                    pageview_rows=("is_pageview_int", "sum"),
                    positive_event_rows=("is_positive_int", "sum"),
                    strict_contact_event_rows=("is_strict_contact_int", "sum"),
                    contact_flag_rows=("is_contact_int", "sum"),
                    dwell_raw_sum_sec=("dwell_raw_sec", "sum"),
                    dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                    dwell_event_count=("has_dwell", "sum"),
                )
                .reset_index()
            )
            write_parquet(g, os.path.join(PARTIAL_TIME_DIR, f"part_{part_id:07d}.parquet"))

        # -----------------------------
        # B. Event type x category
        # -----------------------------
        g = (
            df.groupby(
                ["is_login_group", "event_type_clean", "category", "category_name"],
                observed=True,
                dropna=False,
            )
            .agg(
                rows=("event_type_clean", "size"),
                positive_event_rows=("is_positive_int", "sum"),
                strict_contact_event_rows=("is_strict_contact_int", "sum"),
                contact_flag_rows=("is_contact_int", "sum"),
                dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                dwell_event_count=("has_dwell", "sum"),
            )
            .reset_index()
        )
        write_parquet(g, os.path.join(PARTIAL_EVENT_CAT_DIR, f"part_{part_id:07d}.parquet"))

        # -----------------------------
        # C. City x category
        # -----------------------------
        g = (
            df.groupby(
                ["is_login_group", "city_clean", "category", "category_name"],
                observed=True,
                dropna=False,
            )
            .agg(
                rows=("event_type_clean", "size"),
                pageview_rows=("is_pageview_int", "sum"),
                positive_event_rows=("is_positive_int", "sum"),
                strict_contact_event_rows=("is_strict_contact_int", "sum"),
                contact_flag_rows=("is_contact_int", "sum"),
                dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                dwell_event_count=("has_dwell", "sum"),
            )
            .reset_index()
        )
        write_parquet(g, os.path.join(PARTIAL_CITY_CAT_DIR, f"part_{part_id:07d}.parquet"))

        # -----------------------------
        # D. Device x surface
        # -----------------------------
        g = (
            df.groupby(
                ["is_login_group", "device_clean", "surface_clean"],
                observed=True,
                dropna=False,
            )
            .agg(
                rows=("event_type_clean", "size"),
                pageview_rows=("is_pageview_int", "sum"),
                positive_event_rows=("is_positive_int", "sum"),
                strict_contact_event_rows=("is_strict_contact_int", "sum"),
                contact_flag_rows=("is_contact_int", "sum"),
                dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                dwell_event_count=("has_dwell", "sum"),
            )
            .reset_index()
        )
        write_parquet(g, os.path.join(PARTIAL_DEVICE_SURFACE_DIR, f"part_{part_id:07d}.parquet"))

        # -----------------------------
        # E. Strict contact city x category
        # -----------------------------
        contact_df = df[df["event_type_clean"].isin(STRICT_CONTACT_EVENTS)].copy()
        if len(contact_df) > 0:
            g = (
                contact_df.groupby(
                    ["is_login_group", "event_type_clean", "city_clean", "category", "category_name"],
                    observed=True,
                    dropna=False,
                )
                .agg(
                    rows=("event_type_clean", "size"),
                    dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                    dwell_event_count=("has_dwell", "sum"),
                )
                .reset_index()
            )
            write_parquet(g, os.path.join(PARTIAL_CONTACT_CITY_CAT_DIR, f"part_{part_id:07d}.parquet"))

        # -----------------------------
        # F. Session-level SAMPLE
        # -----------------------------
        sess_df = df[df["session_id"].notna()].copy()

        if len(sess_df) > 0:
            sess_df["session_id"] = sess_df["session_id"].astype("string")

            # Only sample session-level to avoid full disk.
            sess_df = sess_df[
                session_sample_mask(
                    sess_df["session_id"],
                    mod=SESSION_SAMPLE_MOD,
                    keep=SESSION_SAMPLE_KEEP,
                )
            ].copy()

            if len(sess_df) > 0:
                sess_df["login_rows"] = (sess_df["is_login_group"] == "login").astype("int8")
                sess_df["non_login_rows"] = (sess_df["is_login_group"] == "non_login").astype("int8")

                sess_df["login_ts"] = sess_df["event_ts_clean"].where(sess_df["is_login_group"] == "login", pd.NaT)
                sess_df["non_login_ts"] = sess_df["event_ts_clean"].where(sess_df["is_login_group"] == "non_login", pd.NaT)

                g = (
                    sess_df.groupby("session_id", observed=True)
                    .agg(
                        first_event_ts=("event_ts_clean", "min"),
                        last_event_ts=("event_ts_clean", "max"),
                        first_active_date=("active_date", "min"),
                        last_active_date=("active_date", "max"),
                        first_login_ts=("login_ts", "min"),
                        first_non_login_ts=("non_login_ts", "min"),

                        event_rows=("event_type_clean", "size"),
                        login_event_rows=("login_rows", "sum"),
                        non_login_event_rows=("non_login_rows", "sum"),

                        pageview_rows=("is_pageview_int", "sum"),
                        positive_event_rows=("is_positive_int", "sum"),
                        strict_contact_event_rows=("is_strict_contact_int", "sum"),
                        contact_flag_rows=("is_contact_int", "sum"),

                        dwell_raw_sum_sec=("dwell_raw_sec", "sum"),
                        dwell_capped_sum_sec=("dwell_capped_sec", "sum"),
                        dwell_event_count=("has_dwell", "sum"),
                    )
                    .reset_index()
                )
                write_parquet(g, os.path.join(PARTIAL_SESSION_DIR, f"part_{part_id:07d}.parquet"))

        part_id += 1

        del df
        gc.collect()

print("\n[DONE MAP]")
print("Partial files:", part_id)

# ============================================================
# 3. REDUCE GLOBAL SUMMARY TABLES
# ============================================================

TIME_GLOB = os.path.join(PARTIAL_TIME_DIR, "*.parquet").replace("\\", "/")
EVENT_CAT_GLOB = os.path.join(PARTIAL_EVENT_CAT_DIR, "*.parquet").replace("\\", "/")
CITY_CAT_GLOB = os.path.join(PARTIAL_CITY_CAT_DIR, "*.parquet").replace("\\", "/")
DEVICE_SURFACE_GLOB = os.path.join(PARTIAL_DEVICE_SURFACE_DIR, "*.parquet").replace("\\", "/")
CONTACT_CITY_CAT_GLOB = os.path.join(PARTIAL_CONTACT_CITY_CAT_DIR, "*.parquet").replace("\\", "/")
SESSION_PARTIAL_GLOB = os.path.join(PARTIAL_SESSION_DIR, "*.parquet").replace("\\", "/")

# -----------------------------
# 3.1 Daily event summary
# -----------------------------
daily_event_summary = run_duck(f"""
SELECT
    active_date,
    is_login_group,

    SUM(event_rows) AS event_rows,
    SUM(pageview_rows) AS pageview_rows,
    SUM(positive_event_rows) AS positive_event_rows,
    SUM(strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(contact_flag_rows) AS contact_flag_rows,

    SUM(dwell_raw_sum_sec) AS dwell_raw_sum_sec,
    SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,
    SUM(dwell_event_count) AS dwell_event_count,

    SUM(positive_event_rows) * 1.0 / NULLIF(SUM(event_rows), 0) AS positive_event_rate,
    SUM(strict_contact_event_rows) * 1.0 / NULLIF(SUM(event_rows), 0) AS strict_contact_event_rate,
    SUM(contact_flag_rows) * 1.0 / NULLIF(SUM(event_rows), 0) AS contact_flag_rate,
    SUM(dwell_capped_sum_sec) * 1.0 / NULLIF(SUM(dwell_event_count), 0) AS avg_dwell_per_dwell_event_sec
FROM read_parquet('{TIME_GLOB}')
GROUP BY active_date, is_login_group
ORDER BY active_date, is_login_group
""", "01_daily_event_summary_by_login_group")

# -----------------------------
# 3.2 Hour summary
# -----------------------------
hour_summary = run_duck(f"""
SELECT
    hour,
    is_login_group,

    SUM(event_rows) AS event_rows,
    SUM(pageview_rows) AS pageview_rows,
    SUM(positive_event_rows) AS positive_event_rows,
    SUM(strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(contact_flag_rows) AS contact_flag_rows,

    SUM(positive_event_rows) * 1.0 / NULLIF(SUM(event_rows), 0) AS positive_event_rate,
    SUM(strict_contact_event_rows) * 1.0 / NULLIF(SUM(event_rows), 0) AS strict_contact_event_rate,
    SUM(contact_flag_rows) * 1.0 / NULLIF(SUM(event_rows), 0) AS contact_flag_rate
FROM read_parquet('{TIME_GLOB}')
WHERE hour >= 0
GROUP BY hour, is_login_group
ORDER BY hour, is_login_group
""", "02_hour_of_day_summary_by_login_group")

# -----------------------------
# 3.3 Weekday-hour summary
# -----------------------------
dow_hour_summary = run_duck(f"""
SELECT
    dow_num,
    dow_name,
    hour,
    is_login_group,

    SUM(event_rows) AS event_rows,
    SUM(pageview_rows) AS pageview_rows,
    SUM(positive_event_rows) AS positive_event_rows,
    SUM(strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(contact_flag_rows) AS contact_flag_rows
FROM read_parquet('{TIME_GLOB}')
WHERE hour >= 0 AND dow_num >= 0
GROUP BY dow_num, dow_name, hour, is_login_group
ORDER BY dow_num, hour, is_login_group
""", "03_weekday_hour_summary_by_login_group")

# -----------------------------
# 3.4 Event x category summary
# -----------------------------
event_category_summary = run_duck(f"""
SELECT
    is_login_group,
    event_type_clean,
    category,
    category_name,

    SUM(rows) AS rows,
    SUM(positive_event_rows) AS positive_event_rows,
    SUM(strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(contact_flag_rows) AS contact_flag_rows,

    SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,
    SUM(dwell_event_count) AS dwell_event_count,

    SUM(positive_event_rows) * 1.0 / NULLIF(SUM(rows), 0) AS positive_event_rate,
    SUM(strict_contact_event_rows) * 1.0 / NULLIF(SUM(rows), 0) AS strict_contact_event_rate,
    SUM(contact_flag_rows) * 1.0 / NULLIF(SUM(rows), 0) AS contact_flag_rate,
    SUM(dwell_capped_sum_sec) * 1.0 / NULLIF(SUM(dwell_event_count), 0) AS avg_dwell_per_dwell_event_sec
FROM read_parquet('{EVENT_CAT_GLOB}')
GROUP BY is_login_group, event_type_clean, category, category_name
ORDER BY rows DESC
""", "04_event_category_summary_by_login_group", show_head=80)

# -----------------------------
# 3.5 City x category summary
# -----------------------------
city_category_summary = run_duck(f"""
SELECT
    is_login_group,
    city_clean,
    category,
    category_name,

    SUM(rows) AS rows,
    SUM(pageview_rows) AS pageview_rows,
    SUM(positive_event_rows) AS positive_event_rows,
    SUM(strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(contact_flag_rows) AS contact_flag_rows,

    SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,
    SUM(dwell_event_count) AS dwell_event_count,

    SUM(positive_event_rows) * 1.0 / NULLIF(SUM(rows), 0) AS positive_event_rate,
    SUM(strict_contact_event_rows) * 1.0 / NULLIF(SUM(rows), 0) AS strict_contact_event_rate,
    SUM(contact_flag_rows) * 1.0 / NULLIF(SUM(rows), 0) AS contact_flag_rate,
    SUM(dwell_capped_sum_sec) * 1.0 / NULLIF(SUM(dwell_event_count), 0) AS avg_dwell_per_dwell_event_sec
FROM read_parquet('{CITY_CAT_GLOB}')
GROUP BY is_login_group, city_clean, category, category_name
ORDER BY rows DESC
""", "05_city_category_summary_by_login_group", show_head=80)

# -----------------------------
# 3.6 Device x surface summary
# -----------------------------
device_surface_summary = run_duck(f"""
SELECT
    is_login_group,
    device_clean,
    surface_clean,

    SUM(rows) AS rows,
    SUM(pageview_rows) AS pageview_rows,
    SUM(positive_event_rows) AS positive_event_rows,
    SUM(strict_contact_event_rows) AS strict_contact_event_rows,
    SUM(contact_flag_rows) AS contact_flag_rows,

    SUM(positive_event_rows) * 1.0 / NULLIF(SUM(rows), 0) AS positive_event_rate,
    SUM(strict_contact_event_rows) * 1.0 / NULLIF(SUM(rows), 0) AS strict_contact_event_rate,
    SUM(contact_flag_rows) * 1.0 / NULLIF(SUM(rows), 0) AS contact_flag_rate
FROM read_parquet('{DEVICE_SURFACE_GLOB}')
GROUP BY is_login_group, device_clean, surface_clean
ORDER BY rows DESC
""", "06_device_surface_summary_by_login_group", show_head=80)

# -----------------------------
# 3.7 Strict contact city-category summary
# -----------------------------
if len(list_glob(CONTACT_CITY_CAT_GLOB)) > 0:
    contact_city_category_summary = run_duck(f"""
    SELECT
        is_login_group,
        event_type_clean,
        city_clean,
        category,
        category_name,

        SUM(rows) AS rows,
        SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,
        SUM(dwell_event_count) AS dwell_event_count,
        SUM(dwell_capped_sum_sec) * 1.0 / NULLIF(SUM(dwell_event_count), 0) AS avg_dwell_per_dwell_event_sec
    FROM read_parquet('{CONTACT_CITY_CAT_GLOB}')
    GROUP BY is_login_group, event_type_clean, city_clean, category, category_name
    ORDER BY rows DESC
    """, "07_contact_city_category_summary_by_login_group", show_head=80)
else:
    contact_city_category_summary = pd.DataFrame()
    print("[SKIP] No strict contact files found.")

# ============================================================
# 4. REDUCE SESSION SAMPLE
# ============================================================

SESSION_SUMMARY_PATH = os.path.join(CACHE_DIR, "session_summary_sample.parquet").replace("\\", "/")

if len(list_glob(SESSION_PARTIAL_GLOB)) > 0:
    con = duckdb.connect()
    con.execute("PRAGMA threads=2")
    con.execute("PRAGMA memory_limit='6GB'")
    con.execute("PRAGMA preserve_insertion_order=false")

    con.execute(f"""
    COPY (
        WITH sess AS (
            SELECT
                CAST(session_id AS VARCHAR) AS session_id,

                MIN(first_event_ts) AS first_event_ts,
                MAX(last_event_ts) AS last_event_ts,
                MIN(first_active_date) AS first_active_date,
                MAX(last_active_date) AS last_active_date,
                MIN(first_login_ts) AS first_login_ts,
                MIN(first_non_login_ts) AS first_non_login_ts,

                SUM(event_rows) AS event_rows,
                SUM(login_event_rows) AS login_event_rows,
                SUM(non_login_event_rows) AS non_login_event_rows,

                SUM(pageview_rows) AS pageview_rows,
                SUM(positive_event_rows) AS positive_event_rows,
                SUM(strict_contact_event_rows) AS strict_contact_event_rows,
                SUM(contact_flag_rows) AS contact_flag_rows,

                SUM(dwell_raw_sum_sec) AS dwell_raw_sum_sec,
                SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,
                SUM(dwell_event_count) AS dwell_event_count
            FROM read_parquet('{SESSION_PARTIAL_GLOB}')
            GROUP BY session_id
        ),

        scored AS (
            SELECT
                *,

                CASE
                    WHEN login_event_rows > 0 AND non_login_event_rows = 0 THEN 'login_only'
                    WHEN non_login_event_rows > 0 AND login_event_rows = 0 THEN 'non_login_only'
                    WHEN login_event_rows > 0
                     AND non_login_event_rows > 0
                     AND first_non_login_ts <= first_login_ts
                        THEN 'nonlogin_then_login_same_session'
                    WHEN login_event_rows > 0
                     AND non_login_event_rows > 0
                     AND first_login_ts < first_non_login_ts
                        THEN 'login_then_nonlogin_same_session'
                    ELSE 'unknown'
                END AS session_type,

                DATE_DIFF('second', first_event_ts, last_event_ts) AS session_duration_sec,

                dwell_capped_sum_sec * 1.0 / NULLIF(dwell_event_count, 0) AS avg_dwell_per_dwell_event_sec,
                positive_event_rows * 1.0 / NULLIF(event_rows, 0) AS positive_event_rate_in_session,
                strict_contact_event_rows * 1.0 / NULLIF(event_rows, 0) AS strict_contact_event_rate_in_session,
                contact_flag_rows * 1.0 / NULLIF(event_rows, 0) AS contact_flag_rate_in_session,

                EXTRACT(hour FROM first_event_ts) AS first_hour,
                CAST(first_active_date AS DATE) AS session_start_date
            FROM sess
        )

        SELECT *
        FROM scored
    ) TO '{SESSION_SUMMARY_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    con.close()
    print("[SAVE SESSION SAMPLE]", SESSION_SUMMARY_PATH)

    session_type_summary = run_duck(f"""
    SELECT
        session_type,

        COUNT(*) AS sampled_sessions,

        SUM(event_rows) AS event_rows,
        AVG(event_rows) AS avg_event_rows_per_session,
        QUANTILE_CONT(event_rows, 0.50) AS median_event_rows_per_session,
        QUANTILE_CONT(event_rows, 0.90) AS p90_event_rows_per_session,

        AVG(pageview_rows) AS avg_pageview_rows_per_session,
        AVG(positive_event_rows) AS avg_positive_event_rows_per_session,
        AVG(strict_contact_event_rows) AS avg_strict_contact_event_rows_per_session,
        AVG(contact_flag_rows) AS avg_contact_flag_rows_per_session,

        SUM(CASE WHEN positive_event_rows > 0 THEN 1 ELSE 0 END) AS sessions_with_positive,
        SUM(CASE WHEN strict_contact_event_rows > 0 THEN 1 ELSE 0 END) AS sessions_with_strict_contact,
        SUM(CASE WHEN contact_flag_rows > 0 THEN 1 ELSE 0 END) AS sessions_with_contact_flag,

        SUM(CASE WHEN positive_event_rows > 0 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS positive_session_rate,
        SUM(CASE WHEN strict_contact_event_rows > 0 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS strict_contact_session_rate,
        SUM(CASE WHEN contact_flag_rows > 0 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS contact_flag_session_rate,

        AVG(dwell_capped_sum_sec) AS avg_total_dwell_capped_sec_per_session,
        QUANTILE_CONT(dwell_capped_sum_sec, 0.50) AS median_total_dwell_capped_sec_per_session,
        QUANTILE_CONT(dwell_capped_sum_sec, 0.75) AS p75_total_dwell_capped_sec_per_session,
        QUANTILE_CONT(dwell_capped_sum_sec, 0.90) AS p90_total_dwell_capped_sec_per_session,
        QUANTILE_CONT(dwell_capped_sum_sec, 0.95) AS p95_total_dwell_capped_sec_per_session,

        AVG(session_duration_sec) AS avg_session_duration_sec,
        QUANTILE_CONT(session_duration_sec, 0.50) AS median_session_duration_sec,
        QUANTILE_CONT(session_duration_sec, 0.90) AS p90_session_duration_sec
    FROM read_parquet('{SESSION_SUMMARY_PATH}')
    GROUP BY session_type
    ORDER BY sampled_sessions DESC
    """, "08_session_type_summary_sample", show_head=50)

    daily_session_timeline = run_duck(f"""
    SELECT
        session_start_date,
        session_type,

        COUNT(*) AS sampled_sessions,
        SUM(event_rows) AS event_rows,
        SUM(pageview_rows) AS pageview_rows,
        SUM(positive_event_rows) AS positive_event_rows,
        SUM(strict_contact_event_rows) AS strict_contact_event_rows,
        SUM(contact_flag_rows) AS contact_flag_rows,
        SUM(dwell_capped_sum_sec) AS dwell_capped_sum_sec,

        AVG(event_rows) AS avg_event_rows_per_session,
        AVG(dwell_capped_sum_sec) AS avg_total_dwell_capped_sec_per_session,
        SUM(CASE WHEN positive_event_rows > 0 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS positive_session_rate,
        SUM(CASE WHEN strict_contact_event_rows > 0 THEN 1 ELSE 0 END) * 1.0 / NULLIF(COUNT(*), 0) AS strict_contact_session_rate
    FROM read_parquet('{SESSION_SUMMARY_PATH}')
    WHERE session_start_date IS NOT NULL
    GROUP BY session_start_date, session_type
    ORDER BY session_start_date, session_type
    """, "09_daily_session_timeline_by_type_sample", show_head=80)

    session_dwell_bucket = run_duck(f"""
    WITH b AS (
        SELECT
            session_type,
            CASE
                WHEN dwell_capped_sum_sec = 0 THEN '00_0_sec'
                WHEN dwell_capped_sum_sec BETWEEN 1 AND 30 THEN '01_1_30_sec'
                WHEN dwell_capped_sum_sec BETWEEN 31 AND 120 THEN '02_31_120_sec'
                WHEN dwell_capped_sum_sec BETWEEN 121 AND 300 THEN '03_2_5_min'
                WHEN dwell_capped_sum_sec BETWEEN 301 AND 900 THEN '04_5_15_min'
                WHEN dwell_capped_sum_sec BETWEEN 901 AND 1800 THEN '05_15_30_min'
                ELSE '06_30plus_min'
            END AS dwell_bucket,

            CASE
                WHEN dwell_capped_sum_sec = 0 THEN 0
                WHEN dwell_capped_sum_sec BETWEEN 1 AND 30 THEN 1
                WHEN dwell_capped_sum_sec BETWEEN 31 AND 120 THEN 2
                WHEN dwell_capped_sum_sec BETWEEN 121 AND 300 THEN 3
                WHEN dwell_capped_sum_sec BETWEEN 301 AND 900 THEN 4
                WHEN dwell_capped_sum_sec BETWEEN 901 AND 1800 THEN 5
                ELSE 6
            END AS sort_key
        FROM read_parquet('{SESSION_SUMMARY_PATH}')
    )

    SELECT
        session_type,
        sort_key,
        dwell_bucket,
        COUNT(*) AS sampled_sessions,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY session_type) AS share_inside_session_type
    FROM b
    GROUP BY session_type, sort_key, dwell_bucket
    ORDER BY session_type, sort_key
    """, "10_session_dwell_bucket_distribution_sample", show_head=100)

else:
    session_type_summary = pd.DataFrame()
    daily_session_timeline = pd.DataFrame()
    session_dwell_bucket = pd.DataFrame()
    print("[SKIP] No sampled session partials.")

# ============================================================
# 5. DERIVED NON-LOGIN TABLES
# ============================================================

# Category summary
nonlogin_cat = (
    event_category_summary[event_category_summary["is_login_group"] == "non_login"]
    .groupby(["category", "category_name"], as_index=False)
    .agg(
        rows=("rows", "sum"),
        positive_event_rows=("positive_event_rows", "sum"),
        strict_contact_event_rows=("strict_contact_event_rows", "sum"),
        contact_flag_rows=("contact_flag_rows", "sum"),
        dwell_capped_sum_sec=("dwell_capped_sum_sec", "sum"),
        dwell_event_count=("dwell_event_count", "sum"),
    )
)

nonlogin_cat["positive_event_rate"] = safe_rate(nonlogin_cat["positive_event_rows"], nonlogin_cat["rows"])
nonlogin_cat["strict_contact_event_rate"] = safe_rate(nonlogin_cat["strict_contact_event_rows"], nonlogin_cat["rows"])
nonlogin_cat["contact_flag_rate"] = safe_rate(nonlogin_cat["contact_flag_rows"], nonlogin_cat["rows"])
nonlogin_cat["avg_dwell_per_dwell_event_sec"] = safe_rate(nonlogin_cat["dwell_capped_sum_sec"], nonlogin_cat["dwell_event_count"])
nonlogin_cat = nonlogin_cat.sort_values("rows", ascending=False)
save_df(nonlogin_cat, "11_nonlogin_category_summary")

# City summary
nonlogin_city_raw = city_category_summary[city_category_summary["is_login_group"] == "non_login"].copy()

nonlogin_city = (
    nonlogin_city_raw
    .groupby("city_clean", as_index=False)
    .agg(
        rows=("rows", "sum"),
        pageview_rows=("pageview_rows", "sum"),
        positive_event_rows=("positive_event_rows", "sum"),
        strict_contact_event_rows=("strict_contact_event_rows", "sum"),
        contact_flag_rows=("contact_flag_rows", "sum"),
        dwell_capped_sum_sec=("dwell_capped_sum_sec", "sum"),
        dwell_event_count=("dwell_event_count", "sum"),
    )
)

nonlogin_city["positive_event_rate"] = safe_rate(nonlogin_city["positive_event_rows"], nonlogin_city["rows"])
nonlogin_city["strict_contact_event_rate"] = safe_rate(nonlogin_city["strict_contact_event_rows"], nonlogin_city["rows"])
nonlogin_city["avg_dwell_per_dwell_event_sec"] = safe_rate(nonlogin_city["dwell_capped_sum_sec"], nonlogin_city["dwell_event_count"])
nonlogin_city = nonlogin_city.sort_values("rows", ascending=False)
save_df(nonlogin_city, "12_nonlogin_city_summary", show_head=80)

# Daily non-login conversion from sampled session
if len(daily_session_timeline) > 0:
    daily_pivot = (
        daily_session_timeline
        .pivot_table(
            index="session_start_date",
            columns="session_type",
            values="sampled_sessions",
            aggfunc="sum"
        )
        .fillna(0)
        .reset_index()
    )

    for c in ["non_login_only", "nonlogin_then_login_same_session", "login_then_nonlogin_same_session", "login_only"]:
        if c not in daily_pivot.columns:
            daily_pivot[c] = 0

    daily_pivot["sampled_sessions_with_non_login"] = (
        daily_pivot["non_login_only"]
        + daily_pivot["nonlogin_then_login_same_session"]
        + daily_pivot["login_then_nonlogin_same_session"]
    )

    daily_pivot["nonlogin_to_login_conversion_rate"] = (
        daily_pivot["nonlogin_then_login_same_session"]
        / daily_pivot["sampled_sessions_with_non_login"].replace(0, np.nan)
    )
    daily_pivot["nonlogin_to_login_conversion_rate_pct"] = daily_pivot["nonlogin_to_login_conversion_rate"] * 100
    daily_pivot["session_start_date"] = pd.to_datetime(daily_pivot["session_start_date"])
    daily_pivot["conversion_rate_7d_ma_pct"] = (
        daily_pivot["nonlogin_to_login_conversion_rate_pct"]
        .rolling(7, min_periods=1)
        .mean()
    )

    save_df(daily_pivot, "13_daily_nonlogin_to_login_conversion_sample")
else:
    daily_pivot = pd.DataFrame()

# ============================================================
# 6. VISUALIZATIONS
# ============================================================

# -----------------------------
# 6.1 Daily event rows: non-login vs login
# -----------------------------
df = daily_event_summary.copy()
df["active_date"] = pd.to_datetime(df["active_date"])
df = df.sort_values(["active_date", "is_login_group"])

fig = px.line(
    plotly_safe_df(df),
    x="active_date",
    y="event_rows",
    color="is_login_group",
    hover_data=[
        "pageview_rows",
        "positive_event_rows",
        "strict_contact_event_rows",
        "contact_flag_rows",
        "positive_event_rate",
        "strict_contact_event_rate",
        "avg_dwell_per_dwell_event_sec",
    ],
    title="Daily event rows: non-login vs login",
)
fig.update_layout(xaxis_title="Date", yaxis_title="Event rows", height=620)
save_fig(fig, "01_daily_event_rows_nonlogin_vs_login")

# -----------------------------
# 6.2 Daily non-login share
# -----------------------------
pivot = (
    df.pivot_table(index="active_date", columns="is_login_group", values="event_rows", aggfunc="sum")
      .fillna(0)
      .reset_index()
)

if "non_login" not in pivot.columns:
    pivot["non_login"] = 0
if "login" not in pivot.columns:
    pivot["login"] = 0

pivot["total_rows"] = pivot["non_login"] + pivot["login"]
pivot["non_login_share_pct"] = pivot["non_login"] / pivot["total_rows"].replace(0, np.nan) * 100
pivot["non_login_share_pct_7d_ma"] = pivot["non_login_share_pct"].rolling(7, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pivot["active_date"],
    y=pivot["non_login_share_pct"],
    mode="lines",
    name="Daily non-login share",
))
fig.add_trace(go.Scatter(
    x=pivot["active_date"],
    y=pivot["non_login_share_pct_7d_ma"],
    mode="lines",
    name="7D moving avg",
))
fig.update_layout(
    title="Daily non-login share of all event rows",
    xaxis_title="Date",
    yaxis_title="Non-login event rows / total event rows (%)",
    height=620,
)
save_fig(fig, "02_daily_nonlogin_share_of_event_rows")

# -----------------------------
# 6.3 Hour-of-day activity
# -----------------------------
df_hour = hour_summary.copy()
df_hour["event_share_inside_login_group"] = (
    df_hour["event_rows"] / df_hour.groupby("is_login_group")["event_rows"].transform("sum")
)

fig = px.line(
    plotly_safe_df(df_hour),
    x="hour",
    y="event_share_inside_login_group",
    color="is_login_group",
    markers=True,
    hover_data=[
        "event_rows",
        "pageview_rows",
        "positive_event_rows",
        "strict_contact_event_rows",
        "positive_event_rate",
        "strict_contact_event_rate",
    ],
    title="Hour-of-day activity distribution: non-login vs login",
)
fig.update_layout(
    xaxis_title="Hour of day",
    yaxis_title="Share of events inside each group",
    height=600,
)
save_fig(fig, "03_hour_of_day_activity_distribution")

# -----------------------------
# 6.4 Non-login weekday-hour heatmap
# -----------------------------
df_heat = dow_hour_summary[
    (dow_hour_summary["is_login_group"] == "non_login")
    & (dow_hour_summary["hour"] >= 0)
    & (dow_hour_summary["dow_num"] >= 0)
].copy()

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df_heat["dow_name"] = pd.Categorical(df_heat["dow_name"], categories=dow_order, ordered=True)

heat = (
    df_heat.pivot_table(index="dow_name", columns="hour", values="event_rows", aggfunc="sum")
           .reindex(dow_order)
           .fillna(0)
)

fig = px.imshow(
    heat,
    aspect="auto",
    title="Non-login activity heatmap: weekday × hour",
    labels=dict(x="Hour", y="Day of week", color="Event rows"),
)
fig.update_layout(height=560)
save_fig(fig, "04_nonlogin_weekday_hour_heatmap")

# -----------------------------
# 6.5 Non-login event type × category
# -----------------------------
df_ec = event_category_summary[event_category_summary["is_login_group"] == "non_login"].copy()
df_ec = make_label_event_category(df_ec)

plot_df = df_ec.sort_values("rows", ascending=False).head(TOP_N)

fig = px.bar(
    plotly_safe_df(plot_df),
    x="label",
    y="rows",
    color="event_type_clean",
    hover_data=[
        "positive_event_rows",
        "strict_contact_event_rows",
        "contact_flag_rows",
        "positive_event_rate",
        "strict_contact_event_rate",
        "avg_dwell_per_dwell_event_sec",
    ],
    title="Non-login users: event type × category",
)
fig.update_layout(xaxis_title="event_type | category", yaxis_title="Rows", xaxis_tickangle=-35, height=720)
save_fig(fig, "05_nonlogin_event_type_category")

# -----------------------------
# 6.6 Non-login strict contact event × category
# -----------------------------
contact_df = df_ec[df_ec["event_type_clean"].isin(STRICT_CONTACT_EVENTS)].copy()
plot_df = contact_df.sort_values("rows", ascending=False).head(TOP_N)

fig = px.bar(
    plotly_safe_df(plot_df),
    x="label",
    y="rows",
    color="event_type_clean",
    hover_data=[
        "strict_contact_event_rows",
        "contact_flag_rows",
        "avg_dwell_per_dwell_event_sec",
    ],
    title="Non-login strict contact events by category",
)
fig.update_layout(xaxis_title="contact event | category", yaxis_title="Rows", xaxis_tickangle=-35, height=680)
save_fig(fig, "06_nonlogin_contact_event_category")

# -----------------------------
# 6.7 Non-login category summary
# -----------------------------
fig = px.bar(
    plotly_safe_df(nonlogin_cat),
    x="category_name",
    y="rows",
    hover_data=[
        "positive_event_rows",
        "strict_contact_event_rows",
        "positive_event_rate",
        "strict_contact_event_rate",
        "avg_dwell_per_dwell_event_sec",
    ],
    title="Non-login rows by category",
)
fig.update_layout(xaxis_title="Category", yaxis_title="Rows", xaxis_tickangle=-25, height=560)
save_fig(fig, "07_nonlogin_rows_by_category")

fig = px.bar(
    plotly_safe_df(nonlogin_cat),
    x="category_name",
    y=["positive_event_rate", "strict_contact_event_rate", "contact_flag_rate"],
    barmode="group",
    hover_data=["rows", "positive_event_rows", "strict_contact_event_rows", "contact_flag_rows"],
    title="Non-login positive/contact rate by category",
)
fig.update_layout(xaxis_title="Category", yaxis_title="Rate", xaxis_tickangle=-25, height=560)
save_fig(fig, "08_nonlogin_contact_rates_by_category")

# -----------------------------
# 6.8 Top cities
# -----------------------------
plot_df = nonlogin_city.head(TOP_N)

fig = px.bar(
    plotly_safe_df(plot_df),
    x="city_clean",
    y="rows",
    hover_data=[
        "pageview_rows",
        "positive_event_rows",
        "strict_contact_event_rows",
        "positive_event_rate",
        "strict_contact_event_rate",
        "avg_dwell_per_dwell_event_sec",
    ],
    title="Top cities by non-login event rows",
)
fig.update_layout(xaxis_title="City", yaxis_title="Rows", xaxis_tickangle=-35, height=650)
save_fig(fig, "09_top_nonlogin_cities")

top_cities = nonlogin_city.head(15)["city_clean"].tolist()
heat_city_cat = (
    nonlogin_city_raw[nonlogin_city_raw["city_clean"].isin(top_cities)]
    .pivot_table(index="city_clean", columns="category_name", values="rows", aggfunc="sum")
    .fillna(0)
)

fig = px.imshow(
    heat_city_cat,
    aspect="auto",
    title="Non-login rows heatmap: top cities × category",
    labels=dict(x="Category", y="City", color="Rows"),
)
fig.update_layout(height=650)
save_fig(fig, "10_nonlogin_city_category_heatmap")

# -----------------------------
# 6.9 Device × surface
# -----------------------------
df_ds = device_surface_summary[device_surface_summary["is_login_group"] == "non_login"].copy()
df_ds["label"] = df_ds["device_clean"].astype(str) + " | " + df_ds["surface_clean"].astype(str)
plot_df = df_ds.sort_values("rows", ascending=False).head(TOP_N)

fig = px.bar(
    plotly_safe_df(plot_df),
    x="label",
    y="rows",
    color="device_clean",
    hover_data=[
        "pageview_rows",
        "positive_event_rows",
        "strict_contact_event_rows",
        "positive_event_rate",
        "strict_contact_event_rate",
    ],
    title="Non-login rows by device × surface",
)
fig.update_layout(xaxis_title="device | surface", yaxis_title="Rows", xaxis_tickangle=-35, height=680)
save_fig(fig, "11_nonlogin_device_surface")

# -----------------------------
# 6.10 Session summary table
# -----------------------------
if len(session_type_summary) > 0:
    df_s = session_type_summary.copy()
    for c in ["positive_session_rate", "strict_contact_session_rate", "contact_flag_session_rate"]:
        df_s[c + "_pct"] = df_s[c] * 100

    fig = go.Figure(data=[go.Table(
        header=dict(values=list(df_s.columns), align="left"),
        cells=dict(values=[df_s[c] for c in df_s.columns], align="left"),
    )])
    fig.update_layout(
        title=f"Session type summary, sampled {SESSION_SAMPLE_KEEP}%",
        height=520,
    )
    save_fig(fig, "12_session_type_summary_table_sample")

    # Sessions by type
    fig = px.bar(
        plotly_safe_df(df_s.sort_values("sampled_sessions", ascending=False)),
        x="session_type",
        y="sampled_sessions",
        hover_data=[
            "avg_event_rows_per_session",
            "median_event_rows_per_session",
            "p90_event_rows_per_session",
            "avg_total_dwell_capped_sec_per_session",
            "median_total_dwell_capped_sec_per_session",
            "positive_session_rate_pct",
            "strict_contact_session_rate_pct",
        ],
        title=f"Sampled sessions by type ({SESSION_SAMPLE_KEEP}% session sample)",
    )
    fig.update_layout(xaxis_title="Session type", yaxis_title="Sampled sessions", xaxis_tickangle=-20, height=600)
    save_fig(fig, "13_sessions_by_type_sample")

    # Dwell by session type
    fig = px.bar(
        plotly_safe_df(df_s),
        x="session_type",
        y=[
            "avg_total_dwell_capped_sec_per_session",
            "median_total_dwell_capped_sec_per_session",
            "p90_total_dwell_capped_sec_per_session",
        ],
        barmode="group",
        hover_data=[
            "sampled_sessions",
            "avg_event_rows_per_session",
            "positive_session_rate_pct",
            "strict_contact_session_rate_pct",
        ],
        title=f"Total dwell time per session by session type ({SESSION_SAMPLE_KEEP}% sample)",
    )
    fig.update_layout(xaxis_title="Session type", yaxis_title="Dwell time, capped seconds", xaxis_tickangle=-20, height=620)
    save_fig(fig, "14_dwell_time_by_session_type_sample")

    # Interaction intensity
    fig = px.bar(
        plotly_safe_df(df_s),
        x="session_type",
        y=[
            "avg_event_rows_per_session",
            "avg_pageview_rows_per_session",
            "avg_positive_event_rows_per_session",
            "avg_strict_contact_event_rows_per_session",
        ],
        barmode="group",
        hover_data=[
            "sampled_sessions",
            "median_event_rows_per_session",
            "p90_event_rows_per_session",
            "median_total_dwell_capped_sec_per_session",
        ],
        title=f"Interaction intensity per session by session type ({SESSION_SAMPLE_KEEP}% sample)",
    )
    fig.update_layout(xaxis_title="Session type", yaxis_title="Average rows per session", xaxis_tickangle=-20, height=620)
    save_fig(fig, "15_interaction_intensity_by_session_type_sample")

# -----------------------------
# 6.11 Dwell bucket distribution
# -----------------------------
if len(session_dwell_bucket) > 0:
    df_db = session_dwell_bucket.copy()
    df_db["share_inside_session_type_pct"] = df_db["share_inside_session_type"] * 100

    fig = px.bar(
        plotly_safe_df(df_db),
        x="dwell_bucket",
        y="share_inside_session_type_pct",
        color="session_type",
        barmode="group",
        hover_data=["sampled_sessions"],
        title=f"Session dwell bucket distribution by session type ({SESSION_SAMPLE_KEEP}% sample)",
    )
    fig.update_layout(xaxis_title="Total dwell time bucket", yaxis_title="Share inside session type (%)", xaxis_tickangle=-25, height=650)
    save_fig(fig, "16_session_dwell_bucket_distribution_sample")

# -----------------------------
# 6.12 Daily sessions by type
# -----------------------------
if len(daily_session_timeline) > 0:
    df_st = daily_session_timeline.copy()
    df_st["session_start_date"] = pd.to_datetime(df_st["session_start_date"])
    df_st = df_st.sort_values(["session_start_date", "session_type"])

    fig = px.line(
        plotly_safe_df(df_st),
        x="session_start_date",
        y="sampled_sessions",
        color="session_type",
        hover_data=[
            "event_rows",
            "positive_event_rows",
            "strict_contact_event_rows",
            "avg_event_rows_per_session",
            "avg_total_dwell_capped_sec_per_session",
            "positive_session_rate",
            "strict_contact_session_rate",
        ],
        title=f"Daily sampled sessions by session type ({SESSION_SAMPLE_KEEP}% sample)",
    )
    fig.update_layout(xaxis_title="Date", yaxis_title="Sampled sessions", height=650)
    save_fig(fig, "17_daily_sessions_by_type_sample")

# -----------------------------
# 6.13 Non-login to login conversion timeline
# -----------------------------
if len(daily_pivot) > 0:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=daily_pivot["session_start_date"],
        y=daily_pivot["nonlogin_to_login_conversion_rate_pct"],
        mode="lines",
        name="Daily conversion",
    ))
    fig.add_trace(go.Scatter(
        x=daily_pivot["session_start_date"],
        y=daily_pivot["conversion_rate_7d_ma_pct"],
        mode="lines",
        name="7D moving avg",
    ))
    fig.update_layout(
        title=f"Same-session non-login → login conversion rate ({SESSION_SAMPLE_KEEP}% sample)",
        xaxis_title="Date",
        yaxis_title="Converted sessions / sessions with non-login (%)",
        height=620,
    )
    save_fig(fig, "18_nonlogin_to_login_conversion_rate_timeline_sample")

# ============================================================
# 7. SAVE CHART INDEX
# ============================================================

chart_index_df = pd.DataFrame(chart_index)
chart_index_path = os.path.join(OUTPUT_DIR, "99_chart_index.csv")
chart_index_df.to_csv(chart_index_path, index=False, encoding="utf-8-sig")

print("\nDONE.")
print("Output folder:", OUTPUT_DIR)
print("Tables folder:", TABLE_DIR)
print("Figures folder:", FIG_DIR)
print("Chart index:", chart_index_path)
display(chart_index_df)

print("""
MAIN TABLES:
01_daily_event_summary_by_login_group.csv
02_hour_of_day_summary_by_login_group.csv
03_weekday_hour_summary_by_login_group.csv
04_event_category_summary_by_login_group.csv
05_city_category_summary_by_login_group.csv
06_device_surface_summary_by_login_group.csv
07_contact_city_category_summary_by_login_group.csv
08_session_type_summary_sample.csv
09_daily_session_timeline_by_type_sample.csv
10_session_dwell_bucket_distribution_sample.csv
11_nonlogin_category_summary.csv
12_nonlogin_city_summary.csv
13_daily_nonlogin_to_login_conversion_sample.csv

MAIN CHARTS:
01_daily_event_rows_nonlogin_vs_login.html
02_daily_nonlogin_share_of_event_rows.html
03_hour_of_day_activity_distribution.html
04_nonlogin_weekday_hour_heatmap.html
05_nonlogin_event_type_category.html
06_nonlogin_contact_event_category.html
07_nonlogin_rows_by_category.html
08_nonlogin_contact_rates_by_category.html
09_top_nonlogin_cities.html
10_nonlogin_city_category_heatmap.html
11_nonlogin_device_surface.html
13_sessions_by_type_sample.html
14_dwell_time_by_session_type_sample.html
15_interaction_intensity_by_session_type_sample.html
16_session_dwell_bucket_distribution_sample.html
17_daily_sessions_by_type_sample.html
18_nonlogin_to_login_conversion_rate_timeline_sample.html
""")

EVENTS_PATH: /kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2/fact_user_events
Event files: 500
OUTPUT_DIR: /kaggle/working/eda_non_login_behavior
SESSION SAMPLE: 2/100 = 2%

[MAP] file 1/500: datathon_fact_user_events-000000000000.parquet

[MAP] file 2/500: datathon_fact_user_events-000000000001.parquet

[MAP] file 3/500: datathon_fact_user_events-000000000002.parquet

[MAP] file 4/500: datathon_fact_user_events-000000000003.parquet

[MAP] file 5/500: datathon_fact_user_events-000000000004.parquet

[MAP] file 6/500: datathon_fact_user_events-000000000005.parquet

[MAP] file 7/500: datathon_fact_user_events-000000000006.parquet

[MAP] file 8/500: datathon_fact_user_events-000000000007.parquet

[MAP] file 9/500: datathon_fact_user_events-000000000008.parquet

[MAP] file 10/500: datathon_fact_user_events-000000000009.parquet

[MAP] file 11/500: datathon_fact_user_events-000000000010.parquet

[MAP] file 12/500: datathon_fact_user_events-000000000011.parquet

[MAP] file 13/500: data

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[SAVE TABLE] /kaggle/working/eda_non_login_behavior/tables/01_daily_event_summary_by_login_group.csv


,active_date,is_login_group,event_rows,pageview_rows,positive_event_rows,strict_contact_event_rows,contact_flag_rows,dwell_raw_sum_sec,dwell_capped_sum_sec,dwell_event_count,positive_event_rate,strict_contact_event_rate,contact_flag_rate,avg_dwell_per_dwell_event_sec
0,2025-11-09,login,721524.0,397652.0,323872.0,30784.0,323872.0,1.564433e+10,660262964.0,381122.0,0.448872,0.042665,0.448872,1732.418921
1,2025-11-09,non_login,362788.0,110291.0,252497.0,5150.0,252497.0,5.430730e+09,162006270.0,93476.0,0.695990,0.014196,0.695990,1733.132248
2,2025-11-10,login,868961.0,422190.0,446771.0,38199.0,446771.0,1.824543e+10,696648016.0,401648.0,0.514144,0.043959,0.514144,1734.474007
3,2025-11-10,non_login,434770.0,124366.0,310404.0,6437.0,310404.0,6.175723e+09,181020642.0,104798.0,0.713950,0.014806,0.713950,1727.329167
4,2025-11-11,login,863448.0,411096.0,452352.0,37345.0,452352.0,1.786703e+10,677539707.0,391129.0,0.523890,0.043251,0.523890,1732.266610
5,2025-11-11,non_login,412610.0,113904.0,298706.0,6201.0,298706.0,5.892406e+09,167684727.0,96891.0,0.723943,0.015029,0.723943,1730.653280
6,2025-11-12,login,913734.0,409906.0,503828.0,36589.0,503828.0,1.785629e+10,676304545.0,390350.0,0.551395,0.040043,0.551395,1732.559357
7,2025-11-12,non_login,420828.0,115816.0,305012.0,6082.0,305012.0,5.961641e+09,167896073.0,96988.0,0.724790,0.014452,0.724790,1731.101507
8,2025-11-13,login,944823.0,410648.0,534175.0,36206.0,534175.0,1.790594e+10,677486704.0,390996.0,0.565370,0.038320,0.565370,1732.720294
9,2025-11-13,non_login,413327.0,113358.0,299969.0,5884.0,299969.0,5.848984e+09,163874960.0,94531.0,0.725743,0.014236,0.725743,1733.557880


[SAVE TABLE] /kaggle/working/eda_non_login_behavior/tables/02_hour_of_day_summary_by_login_group.csv


...

[SAVE FIG] /kaggle/working/eda_non_login_behavior/figures/17_daily_sessions_by_type_sample.html


[SAVE FIG] /kaggle/working/eda_non_login_behavior/figures/18_nonlogin_to_login_conversion_rate_timeline_sample.html



DONE.
Output folder: /kaggle/working/eda_non_login_behavior
Tables folder: /kaggle/working/eda_non_login_behavior/tables
Figures folder: /kaggle/working/eda_non_login_behavior/figures
Chart index: /kaggle/working/eda_non_login_behavior/99_chart_index.csv


,name,title,html_path
0,01_daily_event_rows_nonlogin_vs_login,01_daily_event_rows_nonlogin_vs_login,/kaggle/working/eda_non_login_behavior/figures...
1,02_daily_nonlogin_share_of_event_rows,02_daily_nonlogin_share_of_event_rows,/kaggle/working/eda_non_login_behavior/figures...
2,03_hour_of_day_activity_distribution,03_hour_of_day_activity_distribution,/kaggle/working/eda_non_login_behavior/figures...
3,04_nonlogin_weekday_hour_heatmap,04_nonlogin_weekday_hour_heatmap,/kaggle/working/eda_non_login_behavior/figures...
4,05_nonlogin_event_type_category,05_nonlogin_event_type_category,/kaggle/working/eda_non_login_behavior/figures...
5,06_nonlogin_contact_event_category,06_nonlogin_contact_event_category,/kaggle/working/eda_non_login_behavior/figures...
6,07_nonlogin_rows_by_category,07_nonlogin_rows_by_category,/kaggle/working/eda_non_login_behavior/figures...
7,08_nonlogin_contact_rates_by_category,08_nonlogin_contact_rates_by_category,/kaggle/working/eda_non_login_behavior/figures...
8,09_top_nonlogin_cities,09_top_nonlogin_cities,/kaggle/working/eda_non_login_behavior/figures...
9,10_nonlogin_city_category_heatmap,10_nonlogin_city_category_heatmap,/kaggle/working/eda_non_login_behavior/figures...



MAIN TABLES:
01_daily_event_summary_by_login_group.csv
02_hour_of_day_summary_by_login_group.csv
03_weekday_hour_summary_by_login_group.csv
04_event_category_summary_by_login_group.csv
05_city_category_summary_by_login_group.csv
06_device_surface_summary_by_login_group.csv
07_contact_city_category_summary_by_login_group.csv
08_session_type_summary_sample.csv
09_daily_session_timeline_by_type_sample.csv
10_session_dwell_bucket_distribution_sample.csv
11_nonlogin_category_summary.csv
12_nonlogin_city_summary.csv
13_daily_nonlogin_to_login_conversion_sample.csv

MAIN CHARTS:
01_daily_event_rows_nonlogin_vs_login.html
02_daily_nonlogin_share_of_event_rows.html
03_hour_of_day_activity_distribution.html
04_nonlogin_weekday_hour_heatmap.html
05_nonlogin_event_type_category.html
06_nonlogin_contact_event_category.html
07_nonlogin_rows_by_category.html
08_nonlogin_contact_rates_by_category.html
09_top_nonlogin_cities.html
10_nonlogin_city_category_heatmap.html
11_nonlogin_device_surface.html
1